# Battery State of Charge Estimation

**State of charge** (SOC) is the 

In [7]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat

load a mat file to check the features and observations

In [8]:
project_dir = Path.cwd() # Set the project directory
data_dir = project_dir / 'dataset' # Set the data directory

mat_0_data = loadmat(data_dir/'BSOC_0_degrees.mat')
print(mat_0_data.keys())
print(f"Shape of features X: {mat_0_data['X'].shape}")
print(f"Shape of target Y: {mat_0_data['Y'].shape}")


dict_keys(['__header__', '__version__', '__globals__', 'X', 'Y'])
Shape of features X: (36001, 3)
Shape of target Y: (1, 36001)


**The output from the above cell describes ths nature of the .mat file**
1. (36001,3) 3600 rows and 3 columns
2. (1, 36001) 1 row and 3600 columns

There is need to reshape the target Y in a 1 dimensional array i.e. (1, 36001) to (36001, 1) when creating the pandas data frame

In [ ]:
#print the first 5 rows of the features and target
print(mat_0_data['X'][:5])

features = ['temp', 'voltage', 'current']

[[25.          3.81279383 14.97490133]
 [24.94266489  3.82554626 14.98682903]
 [24.88562762  3.81477028 14.97525325]
 [24.82889299  3.90814596 15.06785312]
 [24.77246509  3.81618866 14.97514311]]


**Convert mat files to pandas dataframe** 

In [11]:
# a function to convert the .mat data to a pandas DataFrame
def matdata_to_pd(file_name): 
    data = loadmat(data_dir / file_name)
    X = data['X'] 
    Y = data['Y'].reshape(-1)              # reshape the target to be a 1D array
    df = pd.DataFrame(X.astype(float), columns=features) # create a DataFrame  
    df['SOC'] = Y.astype(float)            # add state of charge
    return df

In [12]:
df_n10 = matdata_to_pd('BSOC_n10_degrees.mat') # data collected at ambient temperature of -10 degrees Celsius
df_0 = matdata_to_pd('BSOC_0_degrees.mat') # data collected at ambient temperature of 0 degrees Celsius
df_10 = matdata_to_pd('BSOC_10_degrees.mat') # data collected at ambient temperature of 10 degrees Celsius
df_25 = matdata_to_pd('BSOC_25_degrees.mat') # data collected at ambient temperature of 25 degrees Celsius

# create a single DataFrame with all the data
battery_df = pd.concat([df_n10, df_0, df_10, df_25], ignore_index=True)
battery_df.head()

,temp,voltage,current,SOC
0,25.000000,3.812794,14.974901,0.500000
1,24.841275,3.825451,14.986829,0.500154
2,24.683354,3.814582,14.975253,0.500309
3,24.526241,3.907865,15.067853,0.500463
4,24.369937,3.815818,14.975143,0.500617


In [13]:
battery_df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 144004 entries, 0 to 144003
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   temp     144004 non-null  float64
 1   voltage  144004 non-null  float64
 2   current  144004 non-null  float64
 3   SOC      144004 non-null  float64
dtypes: float64(4)
memory usage: 4.4 MB


**Data visualisation**